# CS781 Phase 4 — Notebook P4_01: Judgment-prediction data, statistics, tokenisation, baselines

No neural network is trained here. The notebook prepares the (large) CJPE data once, so every P4_02 experiment can
start in seconds, and gives the first evidence on *which part of a judgment* predicts the outcome.

| Step | What happens |
|---|---|
| 1 | download `Phase_4_CJPE` (≈ 850 MB) from Hugging Face, detect the format, parse id / text / label |
| 2 | checks: labels are 0/1, unique ids, exact duplicates inside train, **leakage** (same text in train and dev/test) |
| 3 | statistics: label balance per split (class imbalance), document lengths |
| 4 | tokenise every document with the Phase-1 tokenizer → one compact `uint16` token file per split + offsets |
| 5 | baselines on dev with the official metric: majority, TF-IDF + logistic regression on the **first / middle / last** 512 words, head+tail, and the whole document; with the default 0.5 threshold and with a dev-tuned threshold |
| 6 | save everything P4_02 needs |

**Kaggle settings**: Accelerator **None**, Internet **ON**, secret `HF` (optional). RAM: the standard 30 GB is enough.
Run time ≈ 15–25 min (most of it TF-IDF on 37k long documents).
**Input**: the **NB1 output** (`01_data_tokenizer_evalset`) — only for `tokenizer/bpe32768/tokenizer.json`.
**Output**: `/kaggle/working/cs781_p1/p4/data/…` (≈ 0.5 GB) → attach this notebook's output to P4_02.

In [1]:
!pip -q install tokenizers ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 3.9 MB/s eta 0:00:00


In [2]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /tmp/cs781_scratch: free 1148.8 GB / total 8758.4 GB
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
HF token found: True


## 0. Configuration and download

In [3]:
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
try: display
except NameError: display = print

P4_DATA = ROOT / "p4" / "data"; P4_DATA.mkdir(parents=True, exist_ok=True)
P4_REPO_DIR = "Phase_4_CJPE"
P4_FILES = {"train": "Train/train.json", "dev": "Train/dev.json", "test": "Test/test.json"}
TOKENIZER = "bpe32768"
SECTION_WORDS = 512                 # size of the head / middle / tail sections for the TF-IDF baselines
P4_LOCAL = os.environ.get("CS781_P4_LOCAL")

def get_p4_file(rel):
    if P4_LOCAL: return Path(P4_LOCAL) / rel
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, f"{P4_REPO_DIR}/{rel}", repo_type="dataset",
                                token=HF_TOKEN, local_dir=str(SCRATCH / "p4raw")))

PATHS = {k: get_p4_file(v) for k, v in P4_FILES.items()}
for k, p in PATHS.items(): print(f"{k}: {p}  ({os.path.getsize(p)/1e6:.0f} MB)")

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 80.3MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

train: /tmp/cs781_scratch/p4raw/Phase_4_CJPE/Train/train.json  (737 MB)
dev: /tmp/cs781_scratch/p4raw/Phase_4_CJPE/Train/dev.json  (80 MB)
test: /tmp/cs781_scratch/p4raw/Phase_4_CJPE/Test/test.json  (35 MB)


## 1. Parse
Accepts a JSON list of records or a dict `{id: record}`, and the usual field names (`id`/`case_id`/`name`,
`text`/`document`, `label`/`labels`/`gold`). A text stored as a list of strings is joined with spaces.
Ids are kept as strings (the official evaluator requires string ids).

In [4]:
ID_KEYS, TEXT_KEYS, LABEL_KEYS = ("id", "case_id", "name", "doc_id", "uid"), ("text", "document", "case_text", "judgment"), ("label", "labels", "gold", "y", "target")

def first_key(r, keys):
    return next((k for k in keys if k in r), None)

def parse(obj, split):
    recs = list(obj.items()) if isinstance(obj, dict) else [(None, r) for r in obj]
    sample = recs[0][1] if recs else {}
    print(f"{split}: {len(recs)} records; record type {type(sample).__name__}; keys {list(sample)[:8] if isinstance(sample, dict) else '-'}")
    ids, texts, labels = [], [], []
    for k, r in recs:
        if not isinstance(r, dict): r = {"text": r}
        idk, tk, lk = first_key(r, ID_KEYS), first_key(r, TEXT_KEYS), first_key(r, LABEL_KEYS)
        t = r[tk] if tk else ""
        if isinstance(t, list): t = " ".join(map(str, t))
        ids.append(str(r[idk]) if idk else str(k)); texts.append(str(t))
        if split != "test":                                   # test labels are private: never read them
            lab = r[lk] if lk else None
            if isinstance(lab, list): lab = lab[0]
            labels.append(int(lab) if lab is not None else None)
    return ids, texts, (labels if labels else None)

DATA = {}
for split, p in PATHS.items():
    t0 = time.time()
    with open(p, encoding="utf-8") as f: obj = json.load(f)
    DATA[split] = dict(zip(("ids", "texts", "labels"), parse(obj, split)))
    del obj; gc.collect()
    print(f"   parsed in {time.time()-t0:.0f}s; first text: {DATA[split]['texts'][0][:200]!r}")

train: 37387 records; record type dict; keys ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
   parsed in 4s; first text: ' Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09.12.2015 passed by the Division Bench of the High Court 1 dismissing S.T. No.192 of 2014 and other companynected matters'
dev: 3505 records; record type dict; keys ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
   parsed in 1s; first text: 'civil appellate jurisdiction civil appeal number 1709\nof\n1988.\nfrom the judgment and order dated 9.10.1987 of t\nhe\ncentral administrative tribunal chandigarh in appln. t.\n--\n1055 of 1986.\nwith\n slp  c'
test: 1517 records; record type dict; keys ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
   parsed in 0s; first text: 'civil appellate jurisdiction civil appeal number 1257\n nl  of 1985.\nfrom the judgment and order dated 24th januar

## 2. Checks: labels, ids, duplicates, leakage
Exact duplicates are found with a hash of the whitespace-normalised, lower-cased text. Train duplicates with
*conflicting* labels are noise; a test document identical to a training one would be leakage.

In [5]:
# Repeated ids: the training file merges ILDC_multi and ILDC_single, which share cases. Same id + same text ->
# keep one copy; same id + different text -> keep both under "<id>#2", "<id>#3" (train/dev only; test ids must stay unique).
for split, d in DATA.items():
    labs = d["labels"]
    if labs is not None:
        bad = [x for x in labs if x not in (0, 1)]
        assert not bad, f"{split}: non-binary labels {set(bad)}"
    n_rep = len(d["ids"]) - len(set(d["ids"]))
    if n_rep:
        assert split != "test", "test ids are not unique"
        first, keep, same_txt, diff_txt, diff_lab = {}, [], 0, 0, 0
        for i, (did, t) in enumerate(zip(d["ids"], d["texts"])):
            if did not in first: first[did] = [i]; keep.append(i); continue
            if any(" ".join(d["texts"][j].split()) == " ".join(t.split()) for j in first[did]):
                same_txt += 1
                if any(labs[j] != labs[i] for j in first[did]): diff_lab += 1
                continue
            diff_txt += 1; first[did].append(i); d["ids"][i] = f"{did}#{len(first[did])}"; keep.append(i)
        for k in ("ids", "texts", "labels"): d[k] = [d[k][i] for i in keep]
        print(f"{split}: {n_rep} repeated ids -> {same_txt} exact copies dropped ({diff_lab} of them with a different label), "
              f"{diff_txt} different texts kept under a new id")
    assert len(set(d["ids"])) == len(d["ids"])
    print(f"{split}: {len(d['ids'])} docs, ids unique, empty texts: {sum(not t.strip() for t in d['texts'])}")

def h(t): return hash64(" ".join(t.lower().split()))
HASH = {s: [h(t) for t in d["texts"]] for s, d in DATA.items()}
tr_counts = Counter(HASH["train"])
dup_groups = {k: v for k, v in tr_counts.items() if v > 1}
conflict = 0
by_hash = defaultdict(set)
for hh, lab in zip(HASH["train"], DATA["train"]["labels"]): by_hash[hh].add(lab)
conflict = sum(1 for hh in dup_groups if len(by_hash[hh]) > 1)
print(f"train: {sum(v - 1 for v in dup_groups.values())} redundant duplicate docs in {len(dup_groups)} groups; "
      f"{conflict} groups with conflicting labels")
trset = set(HASH["train"])
for s in ("dev", "test"):
    print(f"{s}: {sum(x in trset for x in HASH[s])} docs identical to a training doc")
print("dev ∩ test identical docs:", len(set(HASH["dev"]) & set(HASH["test"])))
print("id overlap train/dev:", len(set(DATA["train"]["ids"]) & set(DATA["dev"]["ids"])),
      "| train/test:", len(set(DATA["train"]["ids"]) & set(DATA["test"]["ids"])))

# keep the first copy of each exact duplicate in train (conflicting groups: majority label, ties dropped)
keep, seen = [], set()
for i, hh in enumerate(HASH["train"]):
    if hh in seen: continue
    seen.add(hh)
    if hh in dup_groups and len(by_hash[hh]) > 1:
        labs = [DATA["train"]["labels"][j] for j, x in enumerate(HASH["train"]) if x == hh]
        c = Counter(labs).most_common()
        if len(c) > 1 and c[0][1] == c[1][1]: continue
        DATA["train"]["labels"][i] = c[0][0]
    keep.append(i)
if len(keep) < len(DATA["train"]["ids"]):
    for k in ("ids", "texts", "labels"): DATA["train"][k] = [DATA["train"][k][i] for i in keep]
print("train docs after de-duplication:", len(DATA["train"]["ids"]))

train: 5082 repeated ids -> 5082 exact copies dropped (0 of them with a different label), 0 different texts kept under a new id
train: 32305 docs, ids unique, empty texts: 0
dev: 994 repeated ids -> 994 exact copies dropped (0 of them with a different label), 0 different texts kept under a new id
dev: 2511 docs, ids unique, empty texts: 0
test: 1517 docs, ids unique, empty texts: 0
train: 653 redundant duplicate docs in 642 groups; 0 groups with conflicting labels
dev: 0 docs identical to a training doc
test: 0 docs identical to a training doc
dev ∩ test identical docs: 1517
id overlap train/dev: 0 | train/test: 0
train docs after de-duplication: 31652


## 3. Statistics: class balance and length
Macro-F1 averages the F1 of class 0 and class 1, so a model biased towards the majority class loses on the other.
If the training prior differs from dev/test, the decision threshold must be re-calibrated on dev.

In [6]:
rows = []
for s, d in DATA.items():
    w = np.array([len(t.split()) for t in d["texts"]])
    r = {"split": s, "docs": len(w), "words_mean": w.mean(), "words_median": np.median(w), "words_p90": np.percentile(w, 90),
         "words_max": w.max()}
    if d["labels"] is not None: r["accepted(1)_%"] = 100 * np.mean(d["labels"])
    rows.append(r); DATA[s]["words"] = w
display(pd.DataFrame(rows).set_index("split").round(1))

# casing: the snippets above show both cased and fully lower-cased documents. If the test documents are (almost) all
# lower-case, every split is lower-cased so that training text looks like test text (LOWERCASE = "auto").
def is_lower(t): s = t[:5000]; return sum(c.isupper() for c in s) < 0.002 * max(1, len(s))
LOWER_SHARE = {s: float(np.mean([is_lower(t) for t in d["texts"]])) for s, d in DATA.items()}
print("share of documents that are lower-case:", {k: round(v, 3) for k, v in LOWER_SHARE.items()})
LOWERCASE = "auto"
DO_LOWER = LOWER_SHARE["test"] > 0.9 if LOWERCASE == "auto" else bool(LOWERCASE)
print("=> lower-case all splits:", DO_LOWER)
if DO_LOWER:
    for d in DATA.values(): d["texts"] = [t.lower() for t in d["texts"]]
print("\nexample (dev, first 600 / last 600 chars):")
print(DATA["dev"]["texts"][0][:600], "\n   ...\n", DATA["dev"]["texts"][0][-600:])

,docs,words_mean,words_median,words_p90,words_max,accepted(1)_%
split,,,,,,
train,31652,3153.8,2030.0,6430.9,412606,41.4
dev,2511,3811.5,2909.0,7210.0,42198,50.1
test,1517,3850.2,2940.0,7302.4,40487,NaN


share of documents that are lower-case: {'train': 0.0, 'dev': 0.993, 'test': 0.989}
=> lower-case all splits: True

example (dev, first 600 / last 600 chars):
civil appellate jurisdiction civil appeal number 1709
of
1988.
from the judgment and order dated 9.10.1987 of t
he
central administrative tribunal chandigarh in appln. t.
--
1055 of 1986.
with
 slp  civil  number 6998 of 1988 

c. mahajan mrs. indu goswami c.v. subba rao
p.
parmeshwaran for the appellant in c.a. number 1709 of 1988.

k.d. namboodary for the petitioner in slp  civil  n
o. 6998 of 1988.

m. ashri and mahabir singh for the respondents. the judgment of the companyrt was delivered by

jagannatha shetty j. the civil appeal by speci
al
leave and the companynected slp raise an importa 
   ...
 de
nt
cannumber therefore be sustained. he was found guilty of t
he
charge framed against him. he
was a party to the fraudulent act for self aggrandisemen
t.
he prepared bogus documents for withdrawal of salary in t
he
name of asho

## 4. Tokenisation → compact token files
Each document → `normalize(text, "collapse_all")` (as in pretraining) → BPE ids. All documents of a split are
concatenated in one `uint16` file; `offsets[i]:offsets[i+1]` are document i's tokens. P4_02 reads any section
(first / last / all chunks) with a memory-mapped slice — no re-tokenisation per experiment.

In [7]:
from tokenizers import Tokenizer
TOK_PATH = find_artifact(f"tokenizer/{TOKENIZER}/tokenizer.json")
assert TOK_PATH is not None, "attach the NB1 output (tokenizer/bpe32768/tokenizer.json) as an input"
TOK = Tokenizer.from_file(str(TOK_PATH)); print("tokenizer:", TOK_PATH, "| vocab", TOK.get_vocab_size())
assert TOK.get_vocab_size() <= 65535

TOKSTATS = {}
for s, d in DATA.items():
    t0 = time.time(); path = P4_DATA / f"{s}_tokens.bin"; offsets = [0]; n_tok = 0
    with open(str(path) + ".tmp", "wb") as f:
        for i in range(0, len(d["texts"]), 1000):
            encs = TOK.encode_batch([normalize(t, "collapse_all") for t in d["texts"][i:i + 1000]])
            for e in encs:
                a = np.asarray(e.ids, dtype=np.uint16); f.write(a.tobytes()); n_tok += len(a); offsets.append(n_tok)
    os.replace(str(path) + ".tmp", path)
    offsets = np.asarray(offsets, dtype=np.int64); L = np.diff(offsets)
    np.save(P4_DATA / f"{s}_offsets.npy", offsets)
    TOKSTATS[s] = {"docs": len(L), "tokens_total_M": n_tok / 1e6, "tokens_mean": L.mean(), "tokens_median": np.median(L),
                   "tokens_p90": np.percentile(L, 90), "tokens_max": int(L.max()), "> 512": (L > 512).mean(),
                   "> 1024": (L > 1024).mean(), "> 2048": (L > 2048).mean(), "> 4096": (L > 4096).mean(),
                   "> 8192": (L > 8192).mean(), "sec": time.time() - t0}
display(pd.DataFrame(TOKSTATS).T.round(3))
for s, d in DATA.items():
    save_json({"ids": d["ids"], "labels": d["labels"]}, P4_DATA / f"{s}_meta.json")

tokenizer: /kaggle/input/datasets/aaravoswal/cs781-p1-nb1/cs781_p1/tokenizer/bpe32768/tokenizer.json | vocab 32768


,docs,tokens_total_M,tokens_mean,tokens_median,tokens_p90,tokens_max,> 512,> 1024,> 2048,> 4096,> 8192,sec
train,31652.0,123.108,3889.421,2508.0,7913.7,500299.0,0.884,0.772,0.577,0.293,0.094,247.694
dev,2511.0,11.420,4548.102,3465.0,8430.0,53090.0,0.988,0.954,0.783,0.395,0.109,23.586
test,1517.0,6.971,4595.339,3499.0,8561.2,50574.0,0.987,0.951,0.783,0.401,0.114,14.546


## 5. Baselines on dev (official metric) — which section is predictive?
TF-IDF (word 1–2-grams, hashed into 2^20 features) + class-balanced logistic regression, trained on the training documents restricted to:
the **first**, **middle** or **last** 512 words, first 256 + last 256, and the whole document.
Each is scored with the 0.5 threshold and with the threshold that maximises dev macro-F1 ("tuned" — optimistic,
it is chosen on the same data). The ILDC paper found the end of the judgment most informative; this checks it on
our data with a model that has no notion of position.

In [8]:
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression

def macro_f1(y, p): return f1_score(y, p, average="macro", labels=[0, 1], zero_division=0)
def official(y, p):
    return {"macro_f1": macro_f1(y, p), "precision": precision_score(y, p, average="binary", pos_label=1, zero_division=0),
            "recall": recall_score(y, p, average="binary", pos_label=1, zero_division=0)}
def best_threshold(y, prob):
    grid = np.linspace(0.05, 0.95, 91); sc = [macro_f1(y, (prob >= t).astype(int)) for t in grid]
    return float(grid[int(np.argmax(sc))]), float(max(sc))

def section(t, how, n=SECTION_WORDS):
    w = t.split()
    if how == "head": return " ".join(w[:n])
    if how == "tail": return " ".join(w[-n:])
    if how == "middle":
        m = max(0, len(w) // 2 - n // 2); return " ".join(w[m:m + n])
    if how == "head+tail": return " ".join(w[:n // 2] + (w[-n // 2:] if len(w) > n // 2 else []))
    return " ".join(w[:20000])                                    # "full" (capped at 20k words)

ytr, ydv = np.array(DATA["train"]["labels"]), np.array(DATA["dev"]["labels"])
BASE = {"majority(0)": official(ydv, np.zeros_like(ydv)), "all ones(1)": official(ydv, np.ones_like(ydv))}
for how in ("head", "middle", "tail", "head+tail", "full"):
    t0 = time.time()
    # hashing (no vocabulary in RAM — the full documents have tens of millions of distinct bigrams) + TF-IDF weighting
    hv = HashingVectorizer(ngram_range=(1, 2), n_features=2 ** 20, alternate_sign=False, norm=None, dtype=np.float32)
    tf = TfidfTransformer(sublinear_tf=True)
    Xtr = tf.fit_transform(hv.transform(section(t, how).lower() for t in DATA["train"]["texts"]))
    Xdv = tf.transform(hv.transform(section(t, how).lower() for t in DATA["dev"]["texts"]))
    clf = LogisticRegression(C=4.0, max_iter=3000, class_weight="balanced", solver="liblinear").fit(Xtr, ytr)
    prob = clf.predict_proba(Xdv)[:, 1]
    r = official(ydv, (prob >= 0.5).astype(int)); thr, tuned = best_threshold(ydv, prob)
    BASE[f"tfidf_{how}"] = {**r, "macro_f1_tuned": tuned, "threshold": thr}
    print(f"tfidf {how:10s}: macro-F1 {r['macro_f1']:.4f} (tuned {tuned:.4f} @ {thr:.2f}) | {time.time()-t0:.0f}s")
    del Xtr, Xdv, hv, tf, clf; gc.collect()
display(pd.DataFrame(BASE).T.round(4))

tfidf head      : macro-F1 0.5759 (tuned 0.5909 @ 0.42) | 53s
tfidf middle    : macro-F1 0.5832 (tuned 0.6058 @ 0.43) | 51s
tfidf tail      : macro-F1 0.6547 (tuned 0.6683 @ 0.46) | 47s
tfidf head+tail : macro-F1 0.6470 (tuned 0.6628 @ 0.47) | 47s
tfidf full      : macro-F1 0.6721 (tuned 0.6845 @ 0.43) | 223s


,macro_f1,precision,recall,macro_f1_tuned,threshold
majority(0),0.3327,0.0000,0.0000,NaN,NaN
all ones(1),0.3340,0.5014,1.0000,NaN,NaN
tfidf_head,0.5759,0.6193,0.4432,0.5909,0.42
tfidf_middle,0.5832,0.6222,0.4631,0.6058,0.43
tfidf_tail,0.6547,0.7303,0.5163,0.6683,0.46
tfidf_head+tail,0.6470,0.7167,0.5123,0.6628,0.47
tfidf_full,0.6721,0.7299,0.5624,0.6845,0.43


## 6. Save

In [9]:
save_json([{"id": i, "gold": int(g)} for i, g in zip(DATA["dev"]["ids"], DATA["dev"]["labels"])], P4_DATA / "truth_dev.json")
meta = {"tokenizer": TOKENIZER, "splits": {s: len(d["ids"]) for s, d in DATA.items()},
        "lower_case_share": LOWER_SHARE, "lowercased": bool(DO_LOWER),
        "accepted_rate": {s: float(np.mean(d["labels"])) for s, d in DATA.items() if d["labels"] is not None},
        "train_duplicates_removed": int(sum(v - 1 for v in dup_groups.values())), "train_conflicting_dup_groups": int(conflict),
        "leakage_identical_to_train": {s: int(sum(x in trset for x in HASH[s])) for s in ("dev", "test")},
        "tokens": {s: {k: float(v) for k, v in st.items()} for s, st in TOKSTATS.items()},
        "baselines_dev": BASE}
save_json(meta, P4_DATA / "meta.json")
print(json.dumps({k: v for k, v in meta.items() if k != "baselines_dev"}, indent=1))
print("saved:", sorted(f"{p.name} ({p.stat().st_size/1e6:.0f} MB)" for p in P4_DATA.iterdir()))

{
 "tokenizer": "bpe32768",
 "splits": {
  "train": 31652,
  "dev": 2511,
  "test": 1517
 },
 "lower_case_share": {
  "train": 0.0,
  "dev": 0.9932297889287136,
  "test": 0.988793671720501
 },
 "lowercased": true,
 "accepted_rate": {
  "train": 0.41428661695943386,
  "dev": 0.5013938669852648
 },
 "train_duplicates_removed": 653,
 "train_conflicting_dup_groups": 0,
 "leakage_identical_to_train": {
  "dev": 0,
  "test": 0
 },
 "tokens": {
  "train": {
   "docs": 31652.0,
   "tokens_total_M": 123.107943,
   "tokens_mean": 3889.4206685201566,
   "tokens_median": 2508.0,
   "tokens_p90": 7913.700000000004,
   "tokens_max": 500299.0,
   "> 512": 0.8838304056615696,
   "> 1024": 0.7723366611904461,
   "> 2048": 0.5771199292303804,
   "> 4096": 0.2927777075698218,
   "> 8192": 0.09351699734613926,
   "sec": 247.69444513320923
  },
  "dev": {
   "docs": 2511.0,
   "tokens_total_M": 11.420283,
   "tokens_mean": 4548.10155316607,
   "tokens_median": 3465.0,
   "tokens_p90": 8430.0,
   "tokens_ma